In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dropout, Dense, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy, AUC
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\python\keras\engine\training_arrays_v1.py:37: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.23.2)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


In [2]:
# Set seed for reproducibility
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)

In [3]:
# Load the dataset
data_x = pd.read_csv('X_cma.csv')
data_y = pd.read_csv('y_cma.csv')

In [4]:
# Remove the 'Unnamed: 0' column if it exists
if 'Unnamed: 0' in data_x.columns:
    data_x = data_x.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in data_y.columns:
    data_y = data_y.drop(columns=['Unnamed: 0'])

In [5]:
# Feature selection
columns_to_keep = list(data_x.columns[:6]) + [
    'EMA_25_mkt', 'SMA_100_cma', 'EMA_26_rmw', 'EMA_25_smb',
    'EMA_25_umd', 'EMA_12_cma', 'RSI_9_cma', 'EMA_26_hml',
    'RSI_9_rmw', 'RSI_9_mkt', 'RSI_9_smb', 'RSI_14_hml'
]

X_train = data_x[columns_to_keep]
y_train = data_y['Target']

In [6]:
# Convert to numpy arrays
X = X_train.values
y = y_train.values

In [7]:
# Use StandardScaler to scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [8]:
# Convert back to DataFrame for EDA
X_scaled_df = pd.DataFrame(X_scaled, columns=columns_to_keep)

In [9]:
# Perform EDA on the final input matrix
X_scaled_df.describe().T

,count,mean,std,min,25%,50%,75%,max
RET_cma,2129.0,-8.760802e-18,1.000235,-8.293726,-0.517883,-0.006036,0.525959,7.933323
RET_hml,2129.0,6.674897e-18,1.000235,-3.887676,-0.611578,-0.033145,0.587042,4.753445
RET_mkt,2129.0,2.336214e-17,1.000235,-11.363081,-0.448243,0.018535,0.479991,8.588009
RET_rmw,2129.0,1.168107e-17,1.000235,-5.501667,-0.596083,0.006557,0.610159,5.115630
RET_smb,2129.0,0.000000e+00,1.000235,-8.440059,-0.494293,0.030894,0.573561,6.091110
RET_umd,2129.0,-2.085905e-18,1.000235,-13.256709,-0.516088,-0.003167,0.519158,7.499463
EMA_25_mkt,2129.0,1.174782e-15,1.000235,-2.292113,-0.798175,0.276070,0.832795,1.495881
SMA_100_cma,2129.0,7.475884e-16,1.000235,-1.276413,-0.977475,-0.305629,0.709515,2.710751
EMA_26_rmw,2129.0,-1.922370e-15,1.000235,-2.421149,-0.617095,-0.129423,0.472417,3.368184
EMA_25_smb,2129.0,-8.009876e-17,1.000235,-1.976555,-0.907582,0.492530,0.805121,1.280829


In [10]:
# Class weight function
def cwts(dfs):
    c0, c1 = np.bincount(dfs)
    w0 = (1/c0) * (len(dfs)) / 2 
    w1 = (1/c1) * (len(dfs)) / 2 
    return {0: w0, 1: w1}


In [11]:
# Define sliding window parameters
window_size = 600
test_gap = 180
step_size = 1

In [12]:
# Determine the start point to process only the last 500 windows
start_point = len(X) - (500 * step_size) - window_size - test_gap

In [13]:
# Lists to store results
all_test_predictions = []
all_actual_test_values = []
all_prediction_probabilities = []
all_predicted_classes = []
all_predicted_probabilities = []

In [14]:
# Define model creation function
def create_model(hu=20, lookback=10, features=1):
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(LSTM(units=hu, input_shape=(lookback, features), activation='relu', return_sequences=False, name='LSTM1'))
    model.add(Dropout(0.4, name='Dropout1'))
    model.add(Dense(units=1, activation='sigmoid', name='Output'))
    opt = Adam(learning_rate=0.001, epsilon=1e-08, decay=0.0)
    model.compile(optimizer=opt, 
                  loss=BinaryCrossentropy(), 
                  metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])
    return model

In [15]:
# EarlyStopping and ReduceLROnPlateau callbacks
early_stopping = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.2, patience=5, min_lr=0.0001)

In [16]:
# Train and evaluate the model with sliding window
for start in range(start_point, len(X_scaled) - window_size - test_gap, step_size):
    try:
        end = start + window_size
        X_train_window = X_scaled[start:end]
        y_train_window = y[start:end]
        
        # Define test indices
        test_start = end + test_gap
        test_end = test_start + 1
        if test_end > len(X_scaled):
            break
        X_test_window = X_scaled[test_start:test_end]
        y_test_window = y[test_start:test_end]

        # Sequence length for training
        seqlen = 10
        # Number of features
        numfeat = X_train_window.shape[1]

        # Generate sequence input for the training dataset
        train_generator = TimeseriesGenerator(X_train_window, y_train_window, length=seqlen, batch_size=64)

        # Calculate custom class weights using y_train_window
        class_weight = cwts(y_train_window)

        # Create a new model for each window
        model = create_model(hu=20, lookback=10, features=numfeat)
        model.compile(loss=BinaryCrossentropy(), optimizer=SGD(learning_rate=1e-3), metrics=[BinaryAccuracy(name='accuracy', threshold=0.5), AUC()])

        model.fit(train_generator, epochs=74, verbose=1, class_weight=class_weight, shuffle=False, callbacks=[early_stopping, reduce_lr])

        # Predict on the single test sample directly
        test_input = np.expand_dims(X_test_window, axis=0)
        prediction = model.predict(test_input)
        prediction_probability = prediction[0][0]
        predicted_class = int(prediction_probability > 0.5)
        print(f'Test Prediction: {prediction_probability:.2f} (Class: {predicted_class})')

        # Store test predictions, actual values, and predicted classes
        all_test_predictions.append(predicted_class)
        all_actual_test_values.append(y_test_window[0])
        all_prediction_probabilities.append(prediction_probability)
        all_predicted_classes.append(predicted_class)

        # Calculate and print confusion matrix
        cm = confusion_matrix([y_test_window[0]], [predicted_class])

    except Exception as e:
        print(f'Error at window {start} to {end}: {e}')
        break



Epoch 1/74


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5197 - auc_1: 0.4813 - loss: 0.7080 - learning_rate: 0.0010
Epoch 2/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4557 - auc_1: 0.4648 - loss: 0.7161 - learning_rate: 0.0010
Epoch 3/74
 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4219 - auc_1: 0.4754 - loss: 0.7787

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5171 - auc_1: 0.4978 - loss: 0.7084 - learning_rate: 0.0010
Epoch 4/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5290 - auc_1: 0.4916 - loss: 0.7037 - learning_rate: 0.0010
Epoch 5/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5298 - auc_1: 0.5018 - loss: 0.7033 - learning_rate: 0.0010
Epoch 6/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5282 - auc_1: 0.5026 - loss: 0.7010 - learning_rate: 0.0010
Epoch 7/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4878 - auc_1: 0.4807 - loss: 0.7129 - learning_rate: 0.0010
Epoch 8/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5144 - auc_1: 0.5068 - loss: 0.7009 - learning_rate: 0.0010
Epoch 9/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5004 - auc_1: 0.5083 - loss: 0.7050 - learning_rate: 0.0010
Epoch 10/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5319 - auc_1: 0.5299 - loss: 0.6930 - learning_rate: 0.0010
Epoch 11/74
10/10 

c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


... (994 outputs left out)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6043 - auc_1: 0.5726 - loss: 0.6236 - learning_rate: 0.0010
Epoch 5/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6094 - auc_1: 0.5629 - loss: 0.6126 - learning_rate: 0.0010
Epoch 6/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6124 - auc_1: 0.5779 - loss: 0.6104 - learning_rate: 0.0010
Epoch 7/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5988 - auc_1: 0.5656 - loss: 0.6174 - learning_rate: 0.0010
Epoch 8/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6061 - auc_1: 0.5680 - loss: 0.6073 - learning_rate: 0.0010
Epoch 9/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6093 - auc_1: 0.5780 - loss: 0.6100 - learning_rate: 0.0010
Epoch 10/74
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6168 - auc_1: 0.5951 - loss: 0.5923 - learning_rate: 0.0010
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
Test Prediction: 0.47 (Class: 0)
Epoch 1/74


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\optimizers\base_optimizer.py:33: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4404 - auc_1: 0.4968 - loss: 0.8645 - learning_rate: 0.0010
Epoch 2/74
 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.0625 - auc_1: 0.0000e+00 - loss: 1.2930Error at window 1076 to 1676: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
MemoryError: 

	 [[{{node PyFunc}}]]
	 [[IteratorGetNext]]
Hint: If you want to see a list of allocated tensors when OOM happens, add report_tensor_allocations_upon_oom to RunOptions for current allocation info. This isn't available when running in Eager mode.
 [Op:__inference_one_step_on_iterator_610842]


c:\Users\tonfo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


In [17]:
# Save the results to files for further analysis
results = {
    "predicted_classes": all_test_predictions,
    "actual_test_values": all_actual_test_values,
    "prediction_probabilities": all_prediction_probabilities
}

results_df = pd.DataFrame(results)
results_df.to_csv('model_results_test.csv', index=False)

In [18]:
# Generate and print the combined classification report
combined_classification_report = classification_report(all_actual_test_values, all_predicted_classes, output_dict=True)
print("Combined Classification Report:")
print(combined_classification_report)

Combined Classification Report:
{'0': {'precision': 0.03636363636363636, 'recall': 0.4444444444444444, 'f1-score': 0.06722689075630252, 'support': 9.0}, '1': {'precision': 0.9572649572649573, 'recall': 0.5137614678899083, 'f1-score': 0.6686567164179105, 'support': 218.0}, 'accuracy': 0.5110132158590308, 'macro avg': {'precision': 0.49681429681429684, 'recall': 0.47910295616717635, 'f1-score': 0.3679418035871065, 'support': 227.0}, 'weighted avg': {'precision': 0.9207534511499269, 'recall': 0.5110132158590308, 'f1-score': 0.6448114810392563, 'support': 227.0}}


In [19]:
# Calculate and print the combined confusion matrix
combined_cm = confusion_matrix(all_actual_test_values, all_predicted_classes)
print(f'Combined Confusion Matrix:\n{combined_cm}')


Combined Confusion Matrix:
[[  4   5]
 [106 112]]


In [20]:
# Save the final model
model.save('SL_model_CMA_test.h5')